# SE4050 Deep Learning Assignment
## Notebook 6: EfficientNetB3 Transfer Learning
### Brain Tumor MRI Classification — Model 4 of 4

**Architecture**: EfficientNetB3 pretrained on ImageNet-1K with a custom classification head  
**Training strategy**: Two-phase — feature extraction followed by selective fine-tuning of the upper 30 layers  
**Justification**: EfficientNet (Tan and Le, 2019) introduces **compound scaling** — simultaneously
scaling network width, depth, and resolution by a fixed ratio derived via neural architecture search.
This produces architectures that are consistently more accurate and parameter-efficient than
manual designs. EfficientNetB3 is the B0 baseline scaled to approximately 12M parameters,
compared to VGG16 (138M) and ResNet50 (25M), while typically matching or exceeding their accuracy.
Additionally, EfficientNet blocks incorporate **Squeeze-and-Excitation (SE) attention modules**
that re-weight channel responses based on global context, which may be particularly effective
for detecting localised anomalies characteristic of different tumor types.

**Prerequisite**: Execute `02_Preprocessing.ipynb` before this notebook.

## Section 0: Environment Setup

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import json, time, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU model      : {torch.cuda.get_device_name(0)}')

DATA_DIR  = Path('preprocessed_data')
MODEL_DIR = Path('saved_models') / 'EfficientNetB3'
RES_DIR   = Path('results')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
BATCH_SIZE    = 32
PHASE1_EPOCHS = 15    # Feature extraction: all backbone layers frozen
PHASE2_EPOCHS = 35    # Fine-tuning: top-30 backbone layers unfrozen
LR_PHASE1     = 1e-4
LR_PHASE2     = 5e-6  # Extra conservative LR; EfficientNet is sensitive to large updates
MODEL_NAME    = 'EfficientNetB3'
N_UNFREEZE    = 30    # Number of backbone layers to unfreeze from the top during Phase 2

print(f'Phase 1 : {PHASE1_EPOCHS} epochs @ LR={LR_PHASE1}')
print(f'Phase 2 : {PHASE2_EPOCHS} epochs @ LR={LR_PHASE2} (top-{N_UNFREEZE} layers unfrozen)')

## Section 1: Load Preprocessed Data with EfficientNetB3 Normalisation

EfficientNetB3 uses the same ImageNet normalisation statistics as VGG16 and ResNet50.
Note that the recommended input resolution for EfficientNetB3 is 300x300, but 224x224
is used here to maintain consistency across all four models in this comparison study,
and because EfficientNet architectures are robust to moderate resolution variations.

In [ ]:
X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
X_val   = np.load(DATA_DIR / 'X_val.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

class_weights_arr    = np.load(DATA_DIR / 'class_weights.npy')
CLASS_NAMES          = np.load(DATA_DIR / 'class_names.npy', allow_pickle=True).tolist()
NUM_CLASSES          = len(CLASS_NAMES)
CLASS_COLORS         = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
class_weights_tensor = torch.FloatTensor(class_weights_arr).to(DEVICE)

# EfficientNet pre-training used the same ImageNet statistics
EFFNET_MEAN = [0.485, 0.456, 0.406]
EFFNET_STD  = [0.229, 0.224, 0.225]

print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')
print(f'Classes: {CLASS_NAMES}')
print(f'ImageNet normalisation: mean={EFFNET_MEAN}, std={EFFNET_STD}')

## Section 2: Dataset and DataLoaders

In [ ]:
class MRIDatasetEfficientNet(Dataset):
    """
    MRI Dataset with EfficientNetB3-specific preprocessing transforms.

    Training: random augmentation + ImageNet normalisation.
    Evaluation: ImageNet normalisation only.

    Note: RandAugment is included for EfficientNet specifically, as the
    original EfficientNet paper (Tan and Le, 2019) reports that RandAugment
    significantly boosts performance, particularly for mid-scale variants (B3-B5).
    """
    TRAIN_TRANSFORMS = T.Compose([
        T.ToPILImage(),
        T.RandomRotation(40),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.3),
        T.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=20, scale=(0.8, 1.2)),
        T.ColorJitter(brightness=0.2, contrast=0.15),
        T.RandAugment(num_ops=2, magnitude=7),  # Stochastic policy-based augmentation
        T.ToTensor(),
        T.Normalize(mean=EFFNET_MEAN, std=EFFNET_STD),
    ])
    EVAL_TRANSFORMS = T.Compose([
        T.ToPILImage(),
        T.ToTensor(),
        T.Normalize(mean=EFFNET_MEAN, std=EFFNET_STD),
    ])

    def __init__(self, X, y, augment=False):
        self.X = (X * 255).astype(np.uint8)
        self.y = y.astype(np.int64)
        self.transform = self.TRAIN_TRANSFORMS if augment else self.EVAL_TRANSFORMS

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.transform(self.X[idx]), self.y[idx]


g = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = DataLoader(MRIDatasetEfficientNet(X_train, y_train, augment=True),
                          BATCH_SIZE, shuffle=True,  num_workers=0, generator=g, pin_memory=True)
val_loader   = DataLoader(MRIDatasetEfficientNet(X_val,   y_val,   augment=False),
                          BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(MRIDatasetEfficientNet(X_test,  y_test,  augment=False),
                          BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print('DataLoaders ready.')

## Section 3: EfficientNetB3 Architecture

### Design Rationale

EfficientNetB3 uses **Mobile Inverted Bottleneck (MBConv)** blocks with:
1. **Depthwise separable convolutions** that factorize standard convolutions into
   depthwise (per-channel spatial filtering) and pointwise (cross-channel mixing) steps,
   drastically reducing parameter count and FLOPs.
2. **Squeeze-and-Excitation (SE)** modules that compute channel importance weights via
   global average pooling and two small FC layers, then rescale channel activations accordingly.
   This is analogous to attention over channels and allows the network to focus on
   diagnostically relevant feature channels.
3. **Stochastic depth** (drop-path) regularisation, where entire residual branches are
   randomly bypassed during training, acting as implicit ensemble learning.

For fine-tuning, we unfreeze the top 30 layers of the backbone. EfficientNet's MBConv
structure means these correspond to the later stages (Stage 6-7), which encode the
most task-specific features.

In [ ]:
class EfficientNetB3TransferModel(nn.Module):
    """
    EfficientNetB3 backbone with a custom classification head for 4-class MRI classification.

    Backbone : EfficientNetB3 feature extractor (MBConv stages 1-7)
    Pooling  : AdaptiveAvgPool2d(1) — Global Average Pooling
    Head     : Linear(1536->512) -> BN -> ReLU -> Dropout(0.4)
               Linear(512->256)  -> ReLU -> Dropout(0.3)
               Linear(256->4)

    EfficientNetB3 produces 1536 feature channels at its output.
    """

    def __init__(self, num_classes: int = 4, freeze_base: bool = True):
        super().__init__()

        # Load EfficientNetB3 with official ImageNet-1K weights
        effnet = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)

        # Extract the convolutional feature extractor; discard the original classifier head
        # torchvision EfficientNet exposes .features (Conv stages) and .classifier (FC head)
        self.features        = effnet.features
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

        if freeze_base:
            for param in self.features.parameters():
                param.requires_grad = False

        # Custom classification head
        # EfficientNetB3 outputs 1536 channels from the final MBConv block
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1536, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )
        # Initialise head with Xavier uniform for consistent gradient magnitudes
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = self.global_avg_pool(x)
        return self.classifier(x)

    def unfreeze_top_n_layers(self, n: int = 30):
        """
        Unfreeze the top-`n` parameter tensors in the backbone feature extractor.

        EfficientNet's MBConv blocks are nested, so counting by named parameters
        is more reliable than indexing by module index. The last `n` parameter
        tensors (in reverse order of their position in the network) correspond to
        the highest-level convolutional stages, which encode the most abstract features.

        Parameters
        ----------
        n : number of parameter tensors to unfreeze from the top of the backbone
        """
        all_params = list(self.features.parameters())
        # Unfreeze the last `n` parameters (highest-level stages)
        for param in all_params[-n:]:
            param.requires_grad = True
        unfrozen = sum(p.numel() for p in all_params[-n:])
        print(f'Top-{n} backbone parameters unfrozen: {unfrozen:,} parameters enabled.')


model = EfficientNetB3TransferModel(num_classes=NUM_CLASSES, freeze_base=True).to(DEVICE)
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters            : {total_p:,}')
print(f'Trainable parameters (Ph.1) : {trainable_p:,}')
with torch.no_grad():
    out = model(torch.randn(2, 3, 224, 224).to(DEVICE))
print(f'Forward pass check          : (2,3,224,224) -> {tuple(out.shape)}')

## Section 4: Training Utilities

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """One training epoch: forward pass, backpropagation, and weight update."""
    model.train()
    ls, cor, tot = 0.0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b); loss = criterion(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ls += loss.item()*len(y_b); cor += (logits.argmax(1)==y_b).sum().item(); tot += len(y_b)
    return ls/tot, cor/tot

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on a DataLoader without gradient computation."""
    model.eval()
    ls, cor, tot = 0.0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b); loss = criterion(logits, y_b)
        ls += loss.item()*len(y_b); cor += (logits.argmax(1)==y_b).sum().item(); tot += len(y_b)
    return ls/tot, cor/tot

class EarlyStopping:
    """Monitors validation loss and halts training when no improvement is observed for `patience` epochs."""
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience, self.min_delta = patience, min_delta
        self.counter = 0; self.best_loss = float('inf'); self.should_stop = False
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0; return True
        self.counter += 1
        if self.counter >= self.patience: self.should_stop = True
        return False

def run_phase(model, tr_loader, va_loader, criterion, optimizer, scheduler,
              epochs, path, label, history=None):
    """
    Execute one training phase. Extends `history` from a prior phase if provided.
    Saves the best checkpoint (lowest validation loss) to `path`.
    """
    hist = history or {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    es   = EarlyStopping(patience=10)
    print(f'\n{"="*65}\n  {label}\n{"="*65}')
    print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | {"Val Loss":>10} | {"Val Acc":>10}')
    print('-' * 56)
    t0 = time.time()
    for ep in range(1, epochs+1):
        tl, ta = train_epoch(model, tr_loader, criterion, optimizer, DEVICE)
        vl, va = evaluate(model,   va_loader, criterion, DEVICE)
        scheduler.step(vl)
        improved = es(vl)
        hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        hist['train_acc'].append(ta);  hist['val_acc'].append(va)
        marker = ' <-- best' if improved else ''
        if improved: torch.save(model.state_dict(), path)
        print(f'{ep:>6} | {tl:>10.4f} | {ta:>9.2%} | {vl:>10.4f} | {va:>9.2%}{marker}')
        if es.should_stop: print(f'Early stopping at epoch {ep}.'); break
    print(f'Duration: {(time.time()-t0)/60:.1f} min | Best val loss: {es.best_loss:.4f}')
    return hist

print('Training utilities defined.')

## Section 5: Phase 1 — Feature Extraction

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
best_path = MODEL_DIR / 'efficientnetb3_best.pt'

# Phase 1 optimiser: updates only the custom classification head
opt_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1, weight_decay=1e-4
)
# AdamW is preferred over Adam for EfficientNet as it applies decoupled weight decay,
# which is consistent with how EfficientNet was originally trained.
sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_p1, mode='min', factor=0.5, patience=5, min_lr=1e-7
)

history = run_phase(
    model, train_loader, val_loader, criterion, opt_p1, sch_p1,
    PHASE1_EPOCHS, best_path,
    'PHASE 1 -- Feature Extraction (backbone frozen, head trained only)'
)

## Section 6: Phase 2 — Fine-Tuning (Top-30 Backbone Layers Unfrozen)

In [ ]:
# Reload best Phase 1 checkpoint before unfreezing and fine-tuning
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.unfreeze_top_n_layers(N_UNFREEZE)

trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable parameters: {trainable_p2:,}')

# Extra-low LR for EfficientNet fine-tuning — EfficientNet weights are
# highly optimised and large gradient steps can cause significant performance regression
opt_p2 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE2, weight_decay=1e-4
)
sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_p2, mode='min', factor=0.5, patience=5, min_lr=1e-9
)

history = run_phase(
    model, train_loader, val_loader, criterion, opt_p2, sch_p2,
    PHASE2_EPOCHS, best_path,
    f'PHASE 2 -- Fine-Tuning (top-{N_UNFREEZE} backbone layers unfrozen)',
    history=history
)

with open(RES_DIR / f'{MODEL_NAME}_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'History saved to results/{MODEL_NAME}_history.json')

## Section 7: Learning Curves

In [ ]:
epochs_trained = len(history['train_loss'])
epochs_ran     = range(1, epochs_trained + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{MODEL_NAME} — Training and Validation Curves', fontsize=13, fontweight='bold')

for ax, tr_key, va_key, ylabel in [
    (axes[0], 'train_loss', 'val_loss', 'Cross-Entropy Loss'),
    (axes[1], 'train_acc',  'val_acc',  'Accuracy (%)'),
]:
    ytr = history[tr_key] if 'loss' in tr_key else [v*100 for v in history[tr_key]]
    yva = history[va_key] if 'loss' in va_key else [v*100 for v in history[va_key]]
    ax.plot(epochs_ran, ytr, 'b-', linewidth=1.5, label='Training')
    ax.plot(epochs_ran, yva, 'r-', linewidth=1.5, label='Validation', alpha=0.85)
    if epochs_trained > PHASE1_EPOCHS:
        ax.axvline(PHASE1_EPOCHS, color='orange', linestyle='--', linewidth=1.5, alpha=0.8,
                   label=f'Fine-tuning start (epoch {PHASE1_EPOCHS})')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel.split(" ")[0]} Curves'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_learning_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 8: Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        logits = model(X_b.to(DEVICE))
        all_probs.extend(torch.softmax(logits, 1).cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y_b.numpy())

all_preds = np.array(all_preds); all_labels = np.array(all_labels); all_probs = np.array(all_probs)

test_acc         = accuracy_score(all_labels, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
roc_auc          = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
total_p          = sum(p.numel() for p in model.parameters())
per_class_acc    = [accuracy_score(all_labels[all_labels==i], all_preds[all_labels==i])
                    for i in range(NUM_CLASSES)]

print('=' * 58)
print(f'  {MODEL_NAME} — Final Test Set Results')
print('=' * 58)
print(f'  Accuracy  (weighted) : {test_acc*100:.2f}%')
print(f'  Precision (weighted) : {prec*100:.2f}%')
print(f'  Recall    (weighted) : {rec*100:.2f}%')
print(f'  F1-Score  (weighted) : {f1*100:.2f}%')
print(f'  ROC-AUC   (OvR,wtd) : {roc_auc:.4f}')
print(f'  Total parameters     : {total_p:,}')
print('=' * 58)
print('\nPer-Class Classification Report:')
print(classification_report(all_labels, all_preds,
      target_names=[c.replace('_',' ').title() for c in CLASS_NAMES], digits=4))

## Section 9: Confusion Matrix and ROC Curves

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
pretty  = [c.replace('_', ' ').title() for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
for ax, data, fmt, title, vmax in [
    (axes[0], cm,      'd',   'Absolute Counts', None),
    (axes[1], cm_norm, '.2f', 'Normalised',      1.0),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=pretty, yticklabels=pretty,
                linewidths=0.5, vmin=0, vmax=vmax,
                annot_kws={'fontsize': 10, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
    ax.set_title(title); ax.tick_params(axis='x', rotation=25); ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(9, 7))
for idx, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, idx], all_probs[:, idx])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{cls.replace("_"," ").title()} (AUC = {auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--', linewidth=1.5, label='Random classifier (AUC = 0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'{MODEL_NAME} — ROC Curves | Weighted AUC = {roc_auc:.4f}', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

per_class_acc_pct = [a*100 for a in per_class_acc]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(pretty, per_class_acc_pct, color=CLASS_COLORS, alpha=0.85, edgecolor='white')
ax.axhline(test_acc*100, color='black', linestyle='--', linewidth=1.5,
           label=f'Overall accuracy ({test_acc*100:.1f}%)')
for bar, val in zip(bars, per_class_acc_pct):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 115); ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{MODEL_NAME} — Per-Class Test Accuracy', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_per_class_accuracy.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 10: Save Metrics

In [ ]:
report_dict = classification_report(all_labels, all_preds,
                                    target_names=CLASS_NAMES, output_dict=True)
metrics = {
    'model_name':         MODEL_NAME,
    'architecture_type':  'EfficientNetB3 Transfer Learning (ImageNet pretrained)',
    'test_accuracy':      float(test_acc),
    'weighted_precision': float(prec),
    'weighted_recall':    float(rec),
    'weighted_f1':        float(f1),
    'roc_auc_weighted':   float(roc_auc),
    'per_class_accuracy': {CLASS_NAMES[i]: float(per_class_acc[i]) for i in range(NUM_CLASSES)},
    'per_class_report':   report_dict,
    'total_params':       total_p,
    'epochs_trained':     epochs_trained,
    'fine_tuning':        True,
    'pretrained':         True,
    'pretrained_on':      'ImageNet-1K',
    'unfreeze_strategy':  f'top-{N_UNFREEZE} backbone parameter tensors (later MBConv stages)',
    'hyperparameters': {
        'batch_size':    BATCH_SIZE,
        'lr_phase1':     LR_PHASE1,
        'lr_phase2':     LR_PHASE2,
        'optimizer':     'AdamW',
        'augmentation':  'Standard + RandAugment(num_ops=2, magnitude=7)',
        'normalization': 'EfficientNet ImageNet (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])',
    },
}
with open(RES_DIR / f'{MODEL_NAME}_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved to results/{MODEL_NAME}_metrics.json')
print(f'Summary: Accuracy={test_acc*100:.2f}% | F1={f1*100:.2f}% | AUC={roc_auc:.4f} | Params={total_p:,}')